# Visualize results

This is a notebook to visualize the results of different metric calculations for different datasets and conditions. We want to see:

**Questions:**
- For each model, how do the scores change as we go further into the model (increasing depth)?
- Do different models have different patterns in how their scores change through the layers, for a given dataset?


In [ ]:
%load_ext autoreload
%autoreload 2

### Imports

In [ ]:
import warnings

# This will ignore all UserWarning messages
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import re
from functools import reduce

from project.utils.strs import linear_probe_metrics_dir, linear_probe_compiled_results_dir, linear_probe_figures_dir

In [ ]:
# Make dirs if they don't exist already
for directory in [linear_probe_metrics_dir, linear_probe_compiled_results_dir, linear_probe_figures_dir]:
    directory.mkdir(parents=True, exist_ok=True)

### Mapping dicts

In [ ]:
datasets = [
    'prot_param',
    'uniprot_peptide',
    'interpro_conserved_site',
    'biomap_localization_prediction',
    'interpro_repeat',
    'biomap_ssp_q3',
    'biomap_ssp_q8',
    'uniprot_secondary_structure',
    'interpro_domain',
    'interpro_family',
    'interpro_homologous_superfamily',
    'uniprot_functional_sites',
    'interpro_binding_site',
    'biomap_metal_ion_binding',
    'interpro_active_site',
    'uniprot_topology',
    'uniprot_post_translational_modification',
    'uniprot_phosphorylation',
    'uniprot_lipidation',
    'GO_cc',
    'GO_mf',
    'GO_bp',
 ]

protein_feature_colormapping = {
    # Baseline 1D (Grey)
    "prot_param": "#708090",                       # SlateGray (Baseline)

    # 1D: Sequence Signals (Orange Flow)
    "uniprot_peptide": "#FFB88C",                   # Light Orange
    "interpro_conserved_site": "#FF8C00",           # DarkOrange
    "biomap_localization_prediction": "#E65100",    # Deep Burnt Orange

    # 2D: Secondary Structure (Blue Flow)
    "interpro_repeat": "#A2D2FF",                   # Light Sky Blue
    "uniprot_secondary_structure": "#5D9CEC",       # Soft Blue
    "biomap_ssp_q3": "#3498DB",                     # Bright Blue
    "biomap_ssp_q8": "#1E3A8A",                     # Deep Royal Blue (Flowing toward 3D)
    
    # 3D: Pure Structural Architecture (Light Green Flow)
    "interpro_homologous_superfamily": "#D1FAE5",  # Mint Cream
    "interpro_family": "#A7F3D0",                   # Pale Emerald
    "interpro_domain": "#6EE7B7",                   # Light Sea Green
    "uniprot_topology": "#34D399",                  # Medium Emerald

    # 3D + System Hybrid: Interactive/Active (Dark Green Flow)
    "uniprot_functional_sites": "#065F46",          # Dark Emerald
    "interpro_binding_site": "#064E3B",             # Deep Hunter Green
    "biomap_metal_ion_binding": "#022C22",          # Near-Black Green
    "interpro_active_site": "#0F172A",              # Deepest Green-Grey
    "uniprot_post_translational_modification": "#1E3A1A", # Dark Forest
    "uniprot_phosphorylation": "#14532D",           # Rich Moss Green
    "uniprot_lipidation": "#064E3B",                # Deep Evergreen

    # 4D+: System-Level Biological Process (Magenta Flow)
    "GO_cc": "#F0ABFC",                             # Light Orchid
    "GO_mf": "#D946EF",                             # Steel Magenta
    "GO_bp": "#701A75",                             # Deep Plum/Grape
}

model_colormapping = {
    # ESM2 Models: Sky Blue Family
    'esm2_8m':    '#BFE5FF', # Very Pale Blue
    'esm2_35m':   '#82C0E9', # Sky Blue
    'esm2_150m':  '#1E78B4', # Steel Blue
    'esm2_650m':  '#004C8C', # Midnight Blue

    # Amplify Models: Orange/Vermillion Family
    'amplify_120m':  '#FFCC80', # Light Orange
    'amplify_350m':  '#E66101', # Burnt Orange/Vermillion

    # SAmplify Models: Green Family (High contrast against Blue/Orange)
    'samplify_120m': '#B2E2E2', # Light Mint
    'samplify_350m': '#006D2C', # Forest Green
}

model_rename_dict = {
    'amplify_120m': 'AMPLIFY 120M',
    'samplify_120m': 'SaAMPLIFY 120M',
    'amplify_350m': 'AMPLIFY 350M',
    'samplify_350m': 'SaAMPLIFY 350M',
    'esm2_8m': 'ESM2 8M',
    'esm2_35m': 'ESM2 35M',
    'esm2_150m': 'ESM2 150M',
    'esm2_650m': 'ESM2 650M'
}

dataset_rename_dict = {
    # Physical & Sequence Properties
    'prot_param': 'Protein Parameters (Physicochemical)',
    'uniprot_peptide': 'Peptide Signal Sequences',
    'biomap_localization_prediction': 'Subcellular Localization',
    
    # Secondary & Local Structure
    'biomap_ssp_q3': 'Secondary Structure (3-class)',
    'biomap_ssp_q8': 'Secondary Structure (8-class)',
    'uniprot_secondary_structure': 'Secondary Structure (UniProt)',
    'uniprot_topology': 'Transmembrane Topology',
    
    # Domains & Families
    'interpro_domain': 'InterPro Domains',
    'interpro_family': 'InterPro Families',
    'interpro_homologous_superfamily': 'Homologous Superfamilies',
    'interpro_repeat': 'Structural Repeats',
    
    # Sites & Modifications
    'interpro_conserved_site': 'Conserved Sites',
    'interpro_binding_site': 'Binding Sites',
    'interpro_active_site': 'Enzymatic Active Sites',
    'uniprot_functional_sites': 'Functional Sites',
    'biomap_metal_ion_binding': 'Metal Ion Binding',
    'uniprot_post_translational_modification': 'PTMs',
    'uniprot_phosphorylation': 'Phosphorylation Sites',
    'uniprot_lipidation': 'Lipidation Sites',
    
    # Gene Ontology
    'GO_cc': 'GO Cellular Component',
    'GO_mf': 'GO Molecular Function',
    'GO_bp': 'GO Biological Process',
}

### Reading in the results

In [ ]:
mean_dfs = []
for dataset in datasets:
    for d_path in linear_probe_metrics_dir.glob(f"*{dataset}*"):
        if ('tiny' not in str(d_path)) and ('medium' not in str(d_path)):
            dataset_dfs = []
            
            # Use d_path here since we used 'd' for the dataframe later
            for f in d_path.iterdir():
                try:
                    # Attempt to read the individual file
                    temp_df = pl.read_parquet(f)
                    dataset_dfs.append(temp_df)
                except Exception as e:
                    # This will catch the "PAR1" error and skip the bad file
                    print(f"Skipping corrupted file {f.name}: {e}")
                    continue 

            if dataset_dfs:
                df = pl.concat(dataset_dfs, how='diagonal_relaxed')
                # Clean columns and append
                clean_df = df.select([c for c in df.columns if ('elementwise' not in c)])
                mean_dfs.append(clean_df)

mean_df = pl.concat(mean_dfs, how='diagonal_relaxed')

# Adjust the dataset nomenclature and extract metadata
mean_df = mean_df.with_columns([
    # 1. Extract '512_cutoff' if present, otherwise default to 'standard' (or null)
    pl.col("dataset")
    .str.extract(r"_(512_cutoff)", 1)
    .fill_null("standard")
    .alias("data_subset"),

    # 2. Extract the split number
    pl.col("dataset")
    .str.extract(r"_split(\d+)$", 1)
    .fill_null("0")
    .alias("split"),

    # 3. Clean the dataset name by removing the split and the cutoff suffixes
    pl.col("dataset")
    .str.replace(r"_512_cutoff", "")  # Remove cutoff
    .str.replace(r"_split\d+$", "")   # Remove split
    .alias("dataset")
]).with_columns(
    pl.col("split").cast(pl.Int64)
)

mean_df = mean_df.filter(
    (pl.col('data_subset') == 'standard'),
    (pl.col('split') == 0))

mean_df = mean_df.unique(subset=['dataset', 'model_name', 'layer_num', 'plm_state', 'plm_embeddings_normalized', 'linear_probe_state', 'control_type', 'fold'])

In [ ]:
mean_df.head()

In [ ]:
# Uncomment to export a copy of the data
mean_df.sort(by=['dataset', 'model_name', 'layer_num', 'plm_state', 'plm_embeddings_normalized', 'linear_probe_state', 'control_type']).write_parquet(linear_probe_compiled_results_dir / 'compiled_mean_results_all_probes_no_checkpoints.parquet.gz')

### Controls for a given model 

In [ ]:
# Harmonize 'score' column for plotting
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained', 'un-trained']
probe_states = ['trained', 'un-trained']
show_controls = ['original', 'scrambled', 'mean', 'l2_normalized', 'random_gaussian'] #
model_order = ['amplify_120m', 'esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m', 'samplify_120m', 'amplify_350m', 'samplify_350m'] #'esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m', amplify_120m',  'samplify_120m', 'amplify_350m', 'samplify_350m'     
dataset_order = [
    'prot_param',
    'uniprot_peptide',
    'interpro_conserved_site',
    'biomap_localization_prediction',
    'interpro_repeat',
    'biomap_ssp_q3',
    'biomap_ssp_q8',
    'uniprot_secondary_structure',
    'interpro_domain',
    'interpro_family',
    'interpro_homologous_superfamily',
    'uniprot_functional_sites',
    'interpro_binding_site',
    'biomap_metal_ion_binding',
    'interpro_active_site',
    'uniprot_topology',
    'uniprot_post_translational_modification',
    'uniprot_phosphorylation',
    # 'uniprot_lipidation',
    'GO_cc',
    'GO_mf',
    'GO_bp',
 ]

xaxis_var = 'layer_num'
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 7
# row_by = 'model_name'
# style_by = 
color_by = 'condition'

standard_cols = mean_df.columns[:8] + ['fold']

for model in model_order:

    # Combine standard columns
    score_dfs = []
    for score in scores_to_plot:
        # Select the intended score column, where there aren't nulls, and return it renamed as 'score
        score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
            pl.col(score).alias('score')
        ).select(standard_cols + ['score']))
    # Combine score dfs for different metrics
    score_df = pl.concat(score_dfs)

    # Filter for what we want to focus on

    all_filters = [
        pl.col('plm_embeddings_normalized') == False, # normalization filter
        pl.col('plm_state').is_in(plm_states), # plm training filter
        pl.col('linear_probe_state').is_in(probe_states), # probe training filter
        (pl.col('model_name') == model), # model filter
        pl.col('dataset').is_in(dataset_order), # dataset filter
        pl.col('control_type').is_in(show_controls), # control filter
        ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')), # untrained filter
    ]

    # Apply filters
    combined_filter = reduce(lambda a, b: a & b, all_filters)

    # Remap names for legibility
    df_filtered = score_df.filter(combined_filter).with_columns(
                pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
                pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
                pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
                pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian embedding', 'l2_normalized': 'normalized embedding'}),
        ).with_columns(
                pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'), # concatenate plm, linear probe and control type
        ).with_columns(
            pl.col("dataset").str.replace_all(dataset_regex, "") # Adjust dataset names for showing in figure
        ).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

    fig = sns.relplot(data = df_filtered, 
        x= xaxis_var, 
        y = 'score',
        col = col_by,
        col_wrap = col_wrap_num,
        # row = row_by,
        col_order = [re.sub(dataset_regex, "", i) for i in dataset_order],
        kind = 'line', 
        markers=True,
        errorbar='sd',
        # style=style_by,
        hue = color_by, 
        height = 2,
        palette = sns.color_palette("tab20", 15),
        facet_kws={'sharey': False},
        )
    fig.set_titles(col_template='{col_name}', row_template='{row_name}')

    residue_tasks = [
        'lipidation', 'topology', 'peptide', 'functional_sites', 
        'phosphorylation', 'secondary_structure', 'ssp_q3', 'ssp_q8', 
        'post_translational_modification'
    ]

    # 1. Strip regex from the category list to match the plot titles
    residue_tasks_cleaned = [re.sub(dataset_regex, "", t) for t in residue_tasks]

    # 2. Iterate through each subplot axis
    for ax in fig.axes.flat:
        title_text = ax.get_title().replace('_', ' ')
        
        # Check if this dataset is a residue-level task
        if title_text in residue_tasks_cleaned:
            # Style for Residue-level: Bold and Blue
            ax.set_title(title_text, 
            # fontweight='bold', 
            fontstyle='italic', 
            color='black', 
            fontsize=10)
        else:
            # Style for Protein-level: Bold and Dark Orange
            ax.set_title(title_text, 
            fontweight='bold', 
            color='black', 
            fontsize=10)

    fig.fig.suptitle(model_rename_dict.get(model))
    
    fig.set_axis_labels("layer #", "score")
    if fig._legend:
        leg = fig._legend
        leg.set_title('Condition')
        plt.setp(leg.get_title(), weight='bold')

    fig.tight_layout()
    fig.savefig(linear_probe_figures_dir / f"probing_{model}_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}_controls.png", dpi=300)
    plt.show()

Focusing on particular datasets only:

In [ ]:
# --- 1. Settings & Initial Harmonization ---
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained', 'un-trained']
probe_states = ['trained', 'un-trained']
show_controls = ['original', 'scrambled', 'mean', 'l2_normalized', 'random_gaussian']
model_order = ['amplify_120m', 'esm2_150m', 'amplify_350m', 'esm2_650m']
dataset_order = [
    "biomap_localization_prediction",
    "biomap_ssp_q3",
    "biomap_metal_ion_binding",
    "interpro_domain",
]

xaxis_var = 'layer_num'
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 2
color_by = 'condition'
residue_tasks = ['lipidation', 'topology', 'peptide', 'functional sites', 'phosphorylation', 'secondary structure', 'ssp q3', 'ssp q8', 'PTMs']

standard_cols = mean_df.columns[:8] + ['fold']

# Pre-calculate combined scores to save time in the loop
score_dfs = []
for score in scores_to_plot:
    score_dfs.append(
        mean_df.select(standard_cols + [score])
        .filter(~pl.col(score).is_null())
        .with_columns(pl.col(score).alias('score'))
        .select(standard_cols + ['score'])
    )
full_score_df = pl.concat(score_dfs)

# --- 2. Loop through Models ---
for model in model_order:
    print(f"Generating plots for: {model}")
    
    # Filtering for the specific model
    all_filters = [
        pl.col('plm_embeddings_normalized') == False,
        pl.col('plm_state').is_in(plm_states),
        pl.col('linear_probe_state').is_in(probe_states),
        pl.col('model_name') == model,  # Filter for current model
        pl.col('dataset').is_in(dataset_order),
        pl.col('control_type').is_in(show_controls),
        ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')),
    ]
    combined_filter = reduce(lambda a, b: a & b, all_filters)

    # Cleaning & Remapping
    df_filtered = full_score_df.filter(combined_filter).with_columns(
        pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}),
        pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
        pl.col('control_type').replace_strict({
            'original': 'original sequence', 
            'scrambled': 'scrambled sequence', 
            'mean': 'mean embedding', 
            'random_gaussian': 'gaussian embedding', 
            'l2_normalized': 'normalized embedding'
        }),
    ).with_columns(
        pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator=' & ').alias('condition'),
    ).with_columns(
        # Remove regex and underscores for plotting labels
        pl.col("dataset").str.replace_all(dataset_regex, "").str.replace_all("_", " ") 
    ).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

    # Match the col_order to the cleaned names
    clean_col_order = [re.sub(dataset_regex, "", i).replace("_", " ") for i in dataset_order]

    # Plotting
    fig = sns.relplot(
        data=df_filtered, 
        x=xaxis_var, 
        y='score',
        col=col_by,
        col_wrap=col_wrap_num,
        col_order=clean_col_order,
        kind='line', 
        markers=True,
        errorbar='sd',
        hue=color_by, 
        height=3,
        palette=sns.color_palette("tab20", 15),
        facet_kws={'sharey': False},
    )

    # --- Formatting Fixes ---
    fig.set_titles(col_template='{col_name}')
    fig.set_axis_labels("layer #", "score")

    # Set x-axis to whole units (integers)
    for ax in fig.axes.flat:
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        
        # Style titles: Italics for residue, Bold for protein
        title_text = ax.get_title()
        is_residue = any(task in title_text.lower() for task in residue_tasks)
        ax.set_title(title_text, fontstyle='italic' if is_residue else 'normal', 
                     fontweight='normal' if is_residue else 'bold', fontsize=10)

    # Bold Legend Title
    if fig._legend:
        leg = fig._legend
        leg.set_title('Condition')
        plt.setp(leg.get_title(), weight='bold')

    # Figure Super Title
    fig.fig.suptitle(model_rename_dict.get(model, model), y=1.05, fontsize=12,)

    fig.tight_layout()
    
    # Save with model-specific filename
    fig.savefig(linear_probe_figures_dir / f"probing_one_dataset_{'_'.join(model_order)}_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}_controls.png", dpi=300)

    plt.show()

### Plotting results for all models

In [ ]:
# Harmonize 'score' column for plotting
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained', ] #'un-trained'
probe_states = ['trained', ] # 'un-trained'
show_controls = ['original'] # 'scrambled', 'mean', 'l2_normalized', 'random_gaussian'
model_order = ['esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m', 'amplify_120m',  'amplify_350m', 'samplify_120m', 'samplify_350m'] #'esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m', 'amplify_120m',  'samplify_120m', 'amplify_350m', 'samplify_350m'
dataset_order = [
    'prot_param',
    'uniprot_peptide',
    'interpro_conserved_site',
    'biomap_localization_prediction',
    'interpro_repeat',
    'biomap_ssp_q3',
    'biomap_ssp_q8',
    'uniprot_secondary_structure',
    'interpro_domain',
    'interpro_family',
    'interpro_homologous_superfamily',
    'uniprot_functional_sites',
    'interpro_binding_site',
    'biomap_metal_ion_binding',
    'interpro_active_site',
    'uniprot_topology',
    'uniprot_post_translational_modification',
    'uniprot_phosphorylation',
    # 'uniprot_lipidation',
    'GO_cc',
    'GO_mf',
    'GO_bp',
 ]

xaxis_var = 'layer_num'
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 7
# row_by = 'model_name'
# style_by = 
color_by = 'model_name'

standard_cols = mean_df.columns[:8] + ['fold']

# Combine standard columns
score_dfs = []
for score in scores_to_plot:
    # Select the intended score column, where there aren't nulls, and return it renamed as 'score
    score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
        pl.col(score).alias('score')
    ).select(standard_cols + ['score']))
# Combine score dfs for different metrics
score_df = pl.concat(score_dfs)

# Filter for what we want to focus on

all_filters = [
    pl.col('plm_embeddings_normalized') == False, # normalization filter
    pl.col('plm_state').is_in(plm_states), # plm training filter
    pl.col('linear_probe_state').is_in(probe_states), # probe training filter
    pl.col('model_name').is_in(model_order), # model filter
    pl.col('dataset').is_in(dataset_order), # dataset filter
    pl.col('control_type').is_in(show_controls), # control filter
    ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')), # untrained filter
]

# Apply filters
combined_filter = reduce(lambda a, b: a & b, all_filters)

# Remap names for legibility
df_filtered = score_df.filter(combined_filter).with_columns(
            pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
            pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
            pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
            pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 'l2_normalized': 'normalized_embedding'}),
    ).with_columns(
            pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'), # concatenate plm, linear probe and control type
    ).with_columns(
        pl.col("dataset").str.replace_all(dataset_regex, "").str.replace_all("_", " ") # Adjust dataset names for showing in figure
    ).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

clean_col_order = [re.sub(dataset_regex, "", i).replace("_", " ") for i in dataset_order]

fig = sns.relplot(data = df_filtered.with_columns(pl.col('model_name').replace_strict(model_rename_dict)), 
    x= xaxis_var, 
    y = 'score',
    col = col_by,
    col_wrap = col_wrap_num,
    # row = row_by,
    col_order = clean_col_order,
    kind = 'line', 
    markers=True,
    errorbar='sd',
    # style=style_by,
    hue = color_by, 
    hue_order=[model_rename_dict.get(m) for m in model_order],
    palette = {model_rename_dict.get(k):v for k,v in model_colormapping.items()},
    height = 2,
    facet_kws={'sharey': False, 'sharex': False},
    )

fig.set_titles(col_template='{col_name}', row_template='{row_name}')

residue_tasks = [
    'lipidation', 'topology', 'peptide', 'functional_sites', 
    'phosphorylation', 'secondary_structure', 'ssp_q3', 'ssp_q8', 
    'post_translational_modification'
]

# 1. Strip regex from the category list to match the plot titles
residue_tasks_cleaned = [re.sub(dataset_regex, "", t).replace("_", " ") for t in residue_tasks]

# 2. Iterate through each subplot axis
for ax in fig.axes.flat:
    title_text = ax.get_title()
    
    # Check if this dataset is a residue-level task
    if title_text in residue_tasks_cleaned:
        # Style for Residue-level: Bold and Blue
        ax.set_title(title_text, 
        # fontweight='bold', 
        fontstyle='italic', 
        color='black', 
        fontsize=10)
    else:
        # Style for Protein-level: Bold and Dark Orange
        ax.set_title(title_text, 
        fontweight='bold', 
        color='black', 
        fontsize=10)

leg = fig._legend
leg.set_title('Model') # Your desired title string
plt.setp(leg.get_title(), weight='bold')

fig.set_axis_labels("layer #", "score")
        
fig.tight_layout()
fig.savefig(linear_probe_figures_dir / f"probing_all_models_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}.png", dpi=300)


### Comparing all models performance with different datasets

In [ ]:
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained']
probe_states = ['trained']
show_controls = ['original']
model_order = ['esm2_150m', 'amplify_120m', 'samplify_120m', 'esm2_650m', 'amplify_350m', 'samplify_350m']
dataset_order = [
    'prot_param', 'uniprot_peptide', 'interpro_conserved_site', 'biomap_localization_prediction',
    'interpro_repeat', 'uniprot_secondary_structure', 'biomap_ssp_q3', 'biomap_ssp_q8',
    'interpro_homologous_superfamily', 'interpro_family', 'interpro_domain', 'uniprot_topology',
    'uniprot_functional_sites', 'interpro_binding_site', 'biomap_metal_ion_binding',
    'interpro_active_site', 'uniprot_post_translational_modification', 'uniprot_phosphorylation',
    'uniprot_lipidation', 'GO_cc', 'GO_mf', 'GO_bp'
]

xaxis_var = 'layer_num'
col_by = 'model_name'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 3
color_by = 'dataset'
standard_cols = mean_df.columns[:8] + ['fold']

# --- 2. Data Harmonization ---
score_dfs = []
for score in scores_to_plot:
    score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
        pl.col(score).alias('score')
    ).select(standard_cols + ['score']))
score_df = pl.concat(score_dfs)

all_filters = [
    pl.col('plm_embeddings_normalized') == False,
    pl.col('plm_state').is_in(plm_states),
    pl.col('linear_probe_state').is_in(probe_states),
    pl.col('model_name').is_in(model_order),
    pl.col('dataset').is_in(dataset_order),
    pl.col('control_type').is_in(show_controls),
    ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')),
]

combined_filter = reduce(lambda a, b: a & b, all_filters)

# --- 3. Name Remapping & Space Cleaning ---
df_filtered = score_df.filter(combined_filter).with_columns(
    pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}),
    pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
    pl.col('control_type').replace_strict({
        'original': 'original sequence', 'scrambled': 'scrambled sequence', 
        'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 
        'l2_normalized': 'normalized_embedding'
    }),
).with_columns(
    pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator=' & ').alias('condition'),
).with_columns(
    # A: Apply dataset dictionary first
    # B: Remove regex prefixes
    # C: Replace all remaining underscores with spaces
    pl.col("dataset").replace(dataset_rename_dict)
    .str.replace_all(dataset_regex, "")
    .str.replace_all("_", " ")
).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

# Helper function to clean a list of names for order/palette consistency
def clean_name(name):
    # Apply dict lookup, then regex, then underscore removal
    mapped = dataset_rename_dict.get(name, name)
    return re.sub(dataset_regex, "", mapped).replace("_", " ")

# Generate order and palette keys using the same logic
clean_hue_order = [clean_name(d) for d in dataset_order]
clean_palette = {clean_name(k): v for k, v in protein_feature_colormapping.items()}

# --- 4. Plotting ---
fig = sns.relplot(
    data=df_filtered.with_columns(pl.col('model_name').replace_strict(model_rename_dict)), 
    x=xaxis_var, 
    y='score',
    col=col_by,
    col_wrap=col_wrap_num,
    col_order=[model_rename_dict.get(m) for m in model_order],
    kind='line', 
    markers=True,
    errorbar='sd',
    hue=color_by, 
    hue_order=clean_hue_order,
    height=3,
    palette=clean_palette,
    facet_kws={'sharey': False, 'sharex': False},
)

# Vertical annotation lines
for ax in fig.axes.flatten():
    ax.axvline(x=12, color='red', linestyle='--', linewidth=1.5, alpha=0.7)

# --- 5. Legend & Titles Styling ---
fig.set_titles(col_template='{col_name}')

fig.set_axis_labels("layer #", "score")
if fig._legend:
    leg = fig._legend
    leg.set_title('Dataset')
    plt.setp(leg.get_title(), weight='bold')

# Italicize residue tasks in legend labels if necessary
# Note: Since the legend is now space-separated, we update the matching list
residue_tasks_cleaned = [clean_name(t) for t in residue_tasks]

fig.tight_layout()
fig.savefig(linear_probe_figures_dir / f"probing_all_models_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}.png", dpi=300)

plt.show()

In [ ]:

scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained']
probe_states = ['trained']
show_controls = ['original']
model_order = ['esm2_150m', 'amplify_120m', 'samplify_120m', 'esm2_650m', 'amplify_350m', 'samplify_350m']
dataset_order = ["prot_param", "uniprot_peptide", "biomap_ssp_q8", "interpro_domain", "GO_mf"]

xaxis_var = 'layer_num'
col_by = 'model_name'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 3
color_by = 'dataset'

standard_cols = mean_df.columns[:8] + ['fold']

# --- 2. Data Harmonization ---
score_dfs = []
for score in scores_to_plot:
    score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
        pl.col(score).alias('score')
    ).select(standard_cols + ['score']))
score_df = pl.concat(score_dfs)

all_filters = [
    pl.col('plm_embeddings_normalized') == False,
    pl.col('plm_state').is_in(plm_states),
    pl.col('linear_probe_state').is_in(probe_states),
    pl.col('model_name').is_in(model_order),
    pl.col('dataset').is_in(dataset_order),
    pl.col('control_type').is_in(show_controls),
    ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')),
]

combined_filter = reduce(lambda a, b: a & b, all_filters)

# --- 3. Cleaning & Dictionary Mapping ---
df_filtered = score_df.filter(combined_filter).with_columns(
    pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}),
    pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
    pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 'l2_normalized': 'normalized_embedding'}),
).with_columns(
    pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'),
).with_columns(
    # APPLY DICTIONARY RENAMING AND UNDERSCORE REMOVAL
    pl.col("dataset").replace(dataset_rename_dict)
    .str.replace_all(dataset_regex, "")
    .str.replace_all("_", " ")
).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

# --- 4. Helper for Consistent Sorting/Colors ---
def clean_name(name):
    mapped = dataset_rename_dict.get(name, name)
    return re.sub(dataset_regex, "", mapped).replace("_", " ")

clean_hue_order = [clean_name(d) for d in dataset_order]
# Re-mapping the palette keys to match the new space-separated names
clean_palette = {clean_name(k): v for k, v in protein_feature_colormapping.items()}

# --- 5. Plotting ---
fig = sns.relplot(
    data = df_filtered.with_columns(pl.col('model_name').replace_strict(model_rename_dict)), 
    x = xaxis_var, 
    y = 'score',
    col = col_by,
    col_wrap = col_wrap_num,
    col_order = [model_rename_dict.get(m) for m in model_order],
    kind = 'line', 
    markers = True,
    errorbar = 'sd',
    hue = color_by, 
    hue_order = clean_hue_order,
    height = 3,
    aspect = 1.5,
    palette = clean_palette,
    facet_kws = {'sharey': False, 'sharex': False},
)

# --- 6. Annotations (Shading & Vertical Lines) ---
for ax in fig.axes.flatten():
    ax.axvline(x=12, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
    # Physical Properties
    ax.axvspan(xmin=0, xmax=1, color='grey', alpha=0.10, zorder=-1)
    # Secondary Structure
    ax.axvspan(xmin=1, xmax=4, color='orange', alpha=0.10, zorder=-1)
    # Tertiary Structure
    ax.axvspan(xmin=4, xmax=12, color='green', alpha=0.10, zorder=-1)

# --- 7. Final Styling ---
# A: Clear "model_name =" from titles
fig.set_titles(col_template='{col_name}')

# B: Set custom X and Y axis labels
fig.set_axis_labels("layer #", "score")

# C: Format Legend
if fig._legend:
    leg = fig._legend
    leg.set_title('Biological Feature')
    plt.setp(leg.get_title(), weight='bold')

fig.tight_layout()

# Save the figure
fig.savefig(linear_probe_figures_dir / f"probing_concept_acquisition_select_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}.png", dpi=300)

plt.show()

### Secondary structure dataset pattern

In [ ]:
cores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained']
probe_states = ['trained']
show_controls = ['original']
model_order = ['esm2_35m','esm2_150m', 'esm2_650m', 'amplify_120m', 'amplify_350m', 'samplify_120m', 'samplify_350m']
dataset_order = ["biomap_ssp_q3", "biomap_ssp_q8", "uniprot_secondary_structure"]

xaxis_var = 'layer_num'
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 3
color_by = 'model_name'

standard_cols = mean_df.columns[:8] + ['fold']

score_dfs = []
for score in scores_to_plot:
    score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
        pl.col(score).alias('score')
    ).select(standard_cols + ['score']))
score_df = pl.concat(score_dfs)

all_filters = [
    pl.col('plm_embeddings_normalized') == False,
    pl.col('plm_state').is_in(plm_states),
    pl.col('linear_probe_state').is_in(probe_states),
    pl.col('model_name').is_in(model_order),
    pl.col('dataset').is_in(dataset_order),
    pl.col('control_type').is_in(show_controls),
    ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')),
]

combined_filter = reduce(lambda a, b: a & b, all_filters)

# --- 2. Cleaning, Renaming & Underscore Removal ---
df_filtered = score_df.filter(combined_filter).with_columns(
    pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}),
    pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
    pl.col('control_type').replace_strict({
        'original': 'original sequence', 'scrambled': 'scrambled sequence', 
        'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 
        'l2_normalized': 'normalized_embedding'
    }),
).with_columns(
    pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'),
).with_columns(
    # Transform dataset names: Map via dict -> strip regex -> replace underscores with spaces
    pl.col("dataset").replace(dataset_rename_dict)
    .str.replace_all(dataset_regex, "")
    .str.replace_all("_", " "),
    # Transform model names for the legend using model_rename_dict
    pl.col("model_name").replace(model_rename_dict)
).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

# Helper for clean list of facet titles
def clean_dataset_name(name):
    mapped = dataset_rename_dict.get(name, name)
    return re.sub(dataset_regex, "", mapped).replace("_", " ")

clean_col_order = [clean_dataset_name(d) for d in dataset_order]
clean_hue_order = [model_rename_dict.get(m, m) for m in model_order]
clean_palette = {model_rename_dict.get(k, k): v for k, v in model_colormapping.items()}

# --- 3. Plotting ---
fig = sns.relplot(
    data = df_filtered, 
    x = xaxis_var, 
    y = 'score',
    col = col_by,
    col_wrap = col_wrap_num,
    col_order = clean_col_order,
    kind = 'line', 
    markers = True,
    errorbar = 'sd',
    hue = color_by, 
    hue_order = clean_hue_order,
    height = 3,
    aspect = 1.5,
    palette = clean_palette,
    facet_kws = {'sharey': False, 'sharex': False},
)

# --- 4. Final Styling & Legend ---
# Clear variable names from facet titles
fig.set_titles(col_template='{col_name}')

# Rename axes
fig.set_axis_labels("layer #", "score")

# Bold the legend title
if fig._legend:
    leg = fig._legend
    leg.set_title('Model')
    plt.setp(leg.get_title(), weight='bold')

# Italicize titles if they represent residue-level tasks
residue_tasks_cleaned = [clean_dataset_name(t) for t in residue_tasks]
for ax in fig.axes.flat:
    title_text = ax.get_title()
    if title_text in residue_tasks_cleaned:
        ax.set_title(title_text, fontstyle='italic', fontsize=10)
    else:
        ax.set_title(title_text, fontweight='bold', fontsize=10)

fig.tight_layout()

# Save figure
fig.savefig(linear_probe_figures_dir / f"probing_ssp_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}.png", dpi=300)
plt.show()

### Model dynamics may be different

In [ ]:
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained']
probe_states = ['trained']
show_controls = ['original']
model_order = ['esm2_150m', 'amplify_120m']
dataset_order = ["interpro_conserved_site", "interpro_domain", "interpro_family"]

xaxis_var = 'layer_num'
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 3
color_by = 'model_name'

standard_cols = mean_df.columns[:8] + ['fold']

# --- 2. Data Harmonization ---
score_dfs = []
for score in scores_to_plot:
    score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
        pl.col(score).alias('score')
    ).select(standard_cols + ['score']))
score_df = pl.concat(score_dfs)

all_filters = [
    pl.col('plm_embeddings_normalized') == False,
    pl.col('plm_state').is_in(plm_states),
    pl.col('linear_probe_state').is_in(probe_states),
    pl.col('model_name').is_in(model_order),
    pl.col('dataset').is_in(dataset_order),
    pl.col('control_type').is_in(show_controls),
    ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')),
    (pl.col('layer_num') <= 15),
]

combined_filter = reduce(lambda a, b: a & b, all_filters)

# --- 3. Cleaning & Dictionary Mapping ---
df_filtered = score_df.filter(combined_filter).with_columns(
    pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}),
    pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
    pl.col('control_type').replace_strict({
        'original': 'original sequence', 'scrambled': 'scrambled sequence', 
        'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 
        'l2_normalized': 'normalized_embedding'
    }),
).with_columns(
    pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'),
).with_columns(
    # Map via dict -> strip regex -> replace underscores with spaces
    pl.col("dataset").replace(dataset_rename_dict)
    .str.replace_all(dataset_regex, "")
    .str.replace_all("_", " "),
    # Rename models for the legend
    pl.col("model_name").replace(model_rename_dict)
).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

# Helpers for consistent mapping
def clean_dataset_name(name):
    mapped = dataset_rename_dict.get(name, name)
    return re.sub(dataset_regex, "", mapped).replace("_", " ")

clean_col_order = [clean_dataset_name(d) for d in dataset_order]
clean_hue_order = [model_rename_dict.get(m, m) for m in model_order]
clean_palette = {model_rename_dict.get(k, k): v for k, v in model_colormapping.items()}

# --- 4. Plotting ---
fig = sns.relplot(
    data = df_filtered, 
    x = xaxis_var, 
    y = 'score',
    col = col_by,
    col_wrap = col_wrap_num,
    col_order = clean_col_order,
    kind = 'line', 
    markers = True,
    errorbar = 'sd',
    hue = color_by, 
    hue_order = clean_hue_order,
    height = 3,
    aspect = 1.25,
    palette = clean_palette,
    facet_kws = {'sharey': False, 'sharex': False},
)

# --- 5. Customizing Axes, Titles & Legend ---

# Force X-axis to use Whole Numbers (Integers)
for ax in fig.axes.flat:
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Clear "dataset =" from titles
fig.set_titles(col_template='{col_name}')

# Style Legend
fig.set_axis_labels("layer #", "score")
if fig._legend:
    leg = fig._legend
    leg.set_title('Model')
    plt.setp(leg.get_title(), weight='bold')

# Specific Subplot Title Styling (Italics for residue tasks)
residue_tasks_cleaned = [clean_dataset_name(t) for t in residue_tasks]
for ax in fig.axes.flat:
    title_text = ax.get_title()
    if title_text in residue_tasks_cleaned:
        ax.set_title(title_text, fontstyle='italic', fontsize=10)
    else:
        ax.set_title(title_text, fontweight='bold', fontsize=10)

fig.tight_layout()

# Save figure
fig.savefig(linear_probe_figures_dir / f"probing_amplify_esm2_comparison_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}.png", dpi=300)
plt.show()

### Effect of scale

In [ ]:
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained']
probe_states = ['trained']
show_controls = ['original']
model_order = ['esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m']
dataset_order = [
    "prot_param", 
    # "interpro_conserved_site", 
    "biomap_ssp_q8",
    # "interpro_domain",
    "GO_mf", 
    # "GO_bp"

]

xaxis_var = 'layer_num' 
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 3
color_by = 'model_name'

standard_cols = mean_df.columns[:8] + ['fold']

# --- 2. Data Harmonization ---
score_dfs = []
for score in scores_to_plot:
    score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
        pl.col(score).alias('score')
    ).select(standard_cols + ['score']))
score_df = pl.concat(score_dfs)

all_filters = [
    pl.col('plm_embeddings_normalized') == False,
    pl.col('plm_state').is_in(plm_states),
    pl.col('linear_probe_state').is_in(probe_states),
    pl.col('model_name').is_in(model_order),
    pl.col('dataset').is_in(dataset_order),
    pl.col('control_type').is_in(show_controls),
    ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')),
]

combined_filter = reduce(lambda a, b: a & b, all_filters)

# --- 3. Cleaning & Transformation (No Mapping) ---
df_filtered = score_df.filter(combined_filter).with_columns(
    pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}),
    pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
    pl.col('control_type').replace_strict({
        'original': 'original sequence', 'scrambled': 'scrambled sequence', 
        'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 
        'l2_normalized': 'normalized_embedding'
    }),
).with_columns(
    pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'),
).with_columns(
    # Strip regex prefixes and replace underscores with spaces ONLY
    pl.col("dataset").str.replace_all(dataset_regex, "").str.replace_all("_", " "),
    # Rename models for the legend
    pl.col("model_name").replace(model_rename_dict)
).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

# Helpers for consistent list generation
def clean_dataset_id(name):
    return re.sub(dataset_regex, "", name).replace("_", " ")

clean_col_order = [clean_dataset_id(d) for d in dataset_order]
clean_hue_order = [model_rename_dict.get(m, m) for m in model_order]
clean_palette = {model_rename_dict.get(k, k): v for k, v in model_colormapping.items()}

# --- 4. Plotting ---
fig = sns.relplot(
    data = df_filtered, 
    x = xaxis_var, 
    y = 'score',
    col = col_by,
    col_wrap = col_wrap_num,
    col_order = clean_col_order,
    kind = 'line', 
    markers = True,
    errorbar = 'sd',
    hue = color_by, 
    hue_order = clean_hue_order,
    height = 3,
    aspect = 1.25,
    palette = clean_palette,
    facet_kws = {'sharey': False, 'sharex': False},
)

# --- 5. Customizing Ticks, Titles & Legend ---

# Force X-axis to use Integers
for ax in fig.axes.flat:
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Clear "dataset =" from titles
fig.set_titles(col_template='{col_name}')

# Set Axis Labels
fig.set_axis_labels("layer #", "score")

# Style Legend
if fig._legend:
    leg = fig._legend
    leg.set_title('Model')
    plt.setp(leg.get_title(), weight='bold')

# Style subplots titles (Italics for residue tasks)
# Ensure residue_tasks list is matched against the cleaned IDs
residue_tasks_cleaned = [clean_dataset_id(t) for t in residue_tasks]
for ax in fig.axes.flat:
    title_text = ax.get_title()
    if title_text in residue_tasks_cleaned:
        ax.set_title(title_text, fontstyle='italic', fontsize=10)
    else:
        ax.set_title(title_text, fontweight='bold', fontsize=10)

fig.tight_layout()

# Save
fig.savefig(linear_probe_figures_dir / f"probing_esm2_scaling_no_mapping.png", dpi=300, bbox_inches='tight')
plt.show()

### Effect of structure awareness

In [ ]:
# Harmonize 'score' column for plotting
scores_to_plot = ['cosine_similarity', 'matthews_corrcoef']
plm_states = ['trained', ] #'un-trained'
probe_states = ['trained', ] # 'un-trained'
show_controls = ['original'] # 'scrambled', 'mean', 'l2_normalized', 'random_gaussian'
model_order = ['amplify_120m', 'amplify_350m', 'samplify_120m', 'samplify_350m'] #'esm2_8m', 'esm2_35m', 'esm2_150m', 'esm2_650m', 'amplify_120m',  'samplify_120m', 'amplify_350m', 'samplify_350m'
dataset_order = [
    'prot_param',
    'uniprot_peptide',
    'interpro_conserved_site',
    'biomap_localization_prediction',
    'interpro_repeat',
    'biomap_ssp_q3',
    'biomap_ssp_q8',
    'uniprot_secondary_structure',
    'interpro_domain',
    'interpro_family',
    'interpro_homologous_superfamily',
    'uniprot_functional_sites',
    'interpro_binding_site',
    'biomap_metal_ion_binding',
    'interpro_active_site',
    'uniprot_topology',
    'uniprot_post_translational_modification',
    'uniprot_phosphorylation',
    'uniprot_lipidation',
    'GO_cc',
    'GO_mf',
    'GO_bp',
 ]

xaxis_var = 'layer_num'
col_by = 'dataset'
dataset_regex = r"(uniprot_|interpro_|biomap_)"
col_wrap_num = 3
# row_by = 'model_name'
# style_by = 
color_by = 'model_name'

standard_cols = mean_df.columns[:8] + ['fold']

# Combine standard columns
score_dfs = []
for score in scores_to_plot:
    # Select the intended score column, where there aren't nulls, and return it renamed as 'score
    score_dfs.append(mean_df.select(standard_cols + [score]).filter(~pl.col(score).is_null()).with_columns(
        pl.col(score).alias('score')
    ).select(standard_cols + ['score']))
# Combine score dfs for different metrics
score_df = pl.concat(score_dfs)

# Filter for what we want to focus on

all_filters = [
    pl.col('plm_embeddings_normalized') == False, # normalization filter
    pl.col('plm_state').is_in(plm_states), # plm training filter
    pl.col('linear_probe_state').is_in(probe_states), # probe training filter
    pl.col('model_name').is_in(model_order), # model filter
    pl.col('dataset').is_in(dataset_order), # dataset filter
    pl.col('control_type').is_in(show_controls), # control filter
    ~((pl.col('plm_state') == 'un-trained') & (pl.col('linear_probe_state') == 'un-trained')), # untrained filter
    # (pl.col('layer_num') <= 15),
]

# Apply filters
combined_filter = reduce(lambda a, b: a & b, all_filters)

# Remap names for legibility
df_filtered = score_df.filter(combined_filter).with_columns(
            pl.col('plm_state').replace_strict({'trained': 'trained PLM', 'un-trained': 'un-trained PLM'}), # renaming for label convenience
            pl.col('linear_probe_state').replace_strict({'trained': 'trained probe', 'naive': 'naive probe', 'un-trained': 'un-trained probe'}),
            pl.col('plm_embeddings_normalized').replace_strict({True: 'embeddings_normalized', False: 'embeddings_not_normalized'}),
            pl.col('control_type').replace_strict({'original': 'original sequence', 'scrambled': 'scrambled sequence', 'mean': 'mean embedding', 'random_gaussian': 'gaussian_embedding', 'l2_normalized': 'normalized_embedding'}),
    ).with_columns(
            pl.concat_str(['plm_state', 'linear_probe_state', 'control_type'], separator= ' & ').alias('condition'), # concatenate plm, linear probe and control type
    ).with_columns(
        pl.col("dataset").str.replace_all(dataset_regex, "") # Adjust dataset names for showing in figure
    ).sort(by=['plm_state', 'linear_probe_state', 'control_type'])

fig = sns.relplot(data = df_filtered, 
    x= xaxis_var, 
    y = 'score',
    col = col_by,
    col_wrap = col_wrap_num,
    # row = row_by,
    col_order = [re.sub(dataset_regex, "", i) for i in dataset_order],
    kind = 'line', 
    markers=True,
    errorbar='sd',
    # style=style_by,
    hue = color_by, 
    hue_order=model_order,
    height = 3,
    aspect = 1.5,
    palette = model_colormapping,
    facet_kws={'sharey': False, 'sharex': False},
    )

fig.set_titles(col_template='{col_name}', row_template='{row_name}')
fig.tight_layout()
fig.savefig(linear_probe_figures_dir / f"structure_awareness_mean_results_{'_'.join(scores_to_plot)}_vs_{xaxis_var}_hue_{color_by}_grid_{col_by}.png", dpi=300)